In [54]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [55]:
df = pd.read_csv('train.txt', sep=';', header=None, names=['Text', 'Emotion'])

In [56]:
df.head()

,Text,Emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [57]:
df.isnull().sum()

Text       0
Emotion    0
dtype: int64

### Converting the emotions into numbers

In [58]:
unique_emotions = df['Emotion'].unique()

In [59]:
unique_emotions

<StringArray>
['sadness', 'anger', 'love', 'surprise', 'fear', 'joy']
Length: 6, dtype: str

In [60]:
emotion_numbers = {}
i = 0
for emo in unique_emotions:
    emotion_numbers[emo] = i
    i += 1

In [61]:
print(emotion_numbers)

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}


In [62]:
df['Emotion'] = df['Emotion'].map(emotion_numbers)

In [63]:
df

,Text,Emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


### Converting all text into lowercase

In [64]:
df['Text'] = df['Text'].apply(lambda x : x.lower())

### Remove punctuations from the texts

In [65]:
import string

def remove_punc(txt):
    return txt.translate(str.maketrans('', '', string.punctuation))

In [66]:
df['Text'] = df['Text'].apply(remove_punc)

### Remove numbers from text

In [67]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['Text'] = df['Text'].apply(remove_numbers)

### Removing emojies and special characters

In [68]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['Text'] = df['Text'].apply(remove_emojis)

### Removing stopwords

In [69]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [70]:
import nltk

In [71]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [72]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to C:\Users\shuvo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shuvo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [73]:
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [74]:
df.loc[1]['Text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [75]:
def remove(txt):
    # words = word_tokenize(txt)
    words = txt.split()
    cleaned = []
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
            
    return ' '.join(cleaned)

In [76]:
df['Text'] = df['Text'].apply(remove)

In [77]:
df.loc[1]['Text']

'go feeling hopeless damned hopeful around someone cares awake'

In [78]:
df.head()

,Text,Emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [79]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['Text'], df['Emotion'], test_size=0.20, random_state=42)

In [80]:
X_train

676      refers course though cant help feeling somehow...
12113                im starting feel im suffering fatigue
7077     feel like probably would liked book little bit...
13005                                  really feel awkward
12123    im feeling little grumpy today lame weather te...
                               ...                        
13418    love leave reader feeling confused slightly de...
5390                                         feel delicate
860                          starting feel little stressed
15795             feel stressed tired worn shape neglected
7270         feel someone rude wrongly done something lose
Name: Text, Length: 12800, dtype: str

In [81]:
X_test

8756                             ive made week feel beaten
4660                              feel strategy worthwhile
6095                     feel worthless weak say want find
304                                        feel clever nov
8241                      im moved ive feeling kind gloomy
                               ...                        
15578    feel useful pulpit find ironic often question ...
5746             dried bladders ready day im feeling brave
6395                             feel thrilled matter days
7624     woke morning text mr c declaring walking work ...
15245                                            feel dumb
Name: Text, Length: 3200, dtype: str

In [82]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Bag of words with Naive Bayes

In [83]:
bow_vectorizer = CountVectorizer()

In [84]:
X_train_bow = bow_vectorizer.fit_transform(X_train)

X_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 116059 stored elements and shape (12800, 13361)>

In [85]:
X_test_bow = bow_vectorizer.transform(X_test)

X_test_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 26936 stored elements and shape (3200, 13361)>

In [86]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [87]:
nb_model = MultinomialNB()

In [88]:
nb_model.fit(X_train_bow, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](6,)","[3720.,1732.,1008., 459.,1540.,4341.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](6,)","[-1.24,-2. ,-2.54,-3.33,-2.12,-1.08]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](6,)","[0,1,2,3,4,5]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](6, 13361)","[[1.,1.,0.,...,0.,0.,1.], [1.,0.,0.,...,0.,0.,0.], [0.,0.,0.,...,0.,1.,0.], [0.,0.,0.,...,0.,0.,0.], [1.,0.,0.,...,0.,0.,0.], [0.,0.,1.,...,1.,2.,0.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](6, 13361)","[[-10.07,-10.07,-10.76,...,-10.76,-10.76,-10.07], [ -9.6 ,-10.29,-10.29,...,-10.29,-10.29,-10.29], [-10.06,-10.06,-10.06,...,-10.06, -9.36,-10.06], [ -9.79, -9.79, -9.79,..., -9.79, -9.79, -9.79], [ -9.53,-10.22,-10.22,...,-10.22,-10.22,-10.22], [-10.91,-10.91,-10.22,...,-10.22, -9.81,-10.91]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,13361


In [89]:
pred_bow = nb_model.predict(X_test_bow)

print(accuracy_score(y_test, pred_bow))

0.768125


# Tf-Idf with Naive Bayes

In [90]:
tfidf_vectorizer = TfidfVectorizer()

In [91]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

X_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 116059 stored elements and shape (12800, 13361)>

In [92]:
X_test_tfidf = tfidf_vectorizer.transform(X_test)

X_test_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 26936 stored elements and shape (3200, 13361)>

In [93]:
nb2_model = MultinomialNB()

nb2_model.fit(X_train_tfidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](6,)","[3720.,1732.,1008., 459.,1540.,4341.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](6,)","[-1.24,-2. ,-2.54,-3.33,-2.12,-1.08]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](6,)","[0,1,2,3,4,5]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](6, 13361)","[[0.5 ,0.48,0. ,...,0. ,0. ,0.31], [0.48,0. ,0. ,...,0. ,0. ,0. ], [0. ,0. ,0. ,...,0. ,0.29,0. ], [0. ,0. ,0. ,...,0. ,0. ,0. ], [0.36,0. ,0. ,...,0. ,0. ,0. ], [0. ,0. ,0.3 ,...,0.26,0.7 ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](6, 13361)","[[ -9.65, -9.67,-10.05,...,-10.05,-10.05, -9.79], [ -9.41, -9.8 , -9.8 ,..., -9.8 , -9.8 , -9.8 ], [ -9.69, -9.69, -9.69,..., -9.69, -9.44, -9.69], [ -9.59, -9.59, -9.59,..., -9.59, -9.59, -9.59], [ -9.47, -9.77, -9.77,..., -9.77, -9.77, -9.77], [-10.14,-10.14, -9.88,..., -9.91, -9.61,-10.14]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,13361


In [94]:
pred_bow = nb2_model.predict(X_test_tfidf)

print(accuracy_score(y_test, pred_bow))

0.6609375


#  Tf-Idf with Logistic Regression

In [95]:
from sklearn.linear_model import LogisticRegression

In [96]:
logistic_model = LogisticRegression(max_iter=1000)

In [97]:
logistic_model.fit(X_train_tfidf, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [98]:
log_pred = logistic_model.predict(X_test_tfidf)

In [99]:
print(accuracy_score(y_test, log_pred))

0.8628125
